In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential, layers, optimizers, losses

In [ ]:
# importing Libraries
import math

from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from eli5.permutation_importance import get_score_importances
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, KBinsDiscretizer
# from sklearn.datasets import make_classification

In [ ]:
from modules.ui import show_hist, show_hists, show_skew_kurt, show_corr, draw_map
from modules.utils import printh, xfrange, scale

In [ ]:
printh("h1", "Load DataFrame")
# main_df = pd.read_csv('default_credit_card_clients.csv')[:9000]
main_df = pd.read_csv('default_credit_card_clients.csv')

printh("h2", "Main DF Info")
print(main_df.info())

printh("h2", "Is NaN")
print("main_df.isna().sum()")
print(main_df.isna().sum())

printh("h2", "List Unique Values in Each Column")
print("main_df.nunique()")
print(main_df.nunique())

printh("h2", "Has Duplicates")
print("main_df.duplicated().sum()")
print(main_df.duplicated().sum())

printh("h2", "Rearranged Columns")
LABEL = list(main_df.columns.values)[-1]
SELECT_FEATURES = list(main_df.columns.values)[:-1]
SELECT_COLUMNS = SELECT_FEATURES + [LABEL]
LABEL = "default"
SELECT_FEATURES = [
  "LIMIT_BAL",
  "SEX",
  "EDUCATION",
  "MARRIAGE",
  "AGE",
  "PAY_0",
  "PAY_2",
  "PAY_3",
  "PAY_4",
  "PAY_5",
  "PAY_6",
  "BILL_AMT1",
  "BILL_AMT2",
  "BILL_AMT3",
  "BILL_AMT4",
  "BILL_AMT5",
  "BILL_AMT6",
  "PAY_AMT1",
  "PAY_AMT2",
  "PAY_AMT3",
  "PAY_AMT4",
  "PAY_AMT5",
  "PAY_AMT6",
]
SELECT_COLUMNS = SELECT_FEATURES + [LABEL]
main_df = main_df[SELECT_COLUMNS]

print(pd.concat([main_df[:5], main_df[-5:]]))

printh("h2", "Features & Label")
print("SELECT_COLUMNS", SELECT_COLUMNS)
print("SELECT_FEATURES", SELECT_FEATURES)
print("LABEL", LABEL)

In [ ]:
def add_features(input_df):
    _input_df = input_df.copy()
    
    _input_df = pd.get_dummies(_input_df, columns=["EDUCATION"], dtype=float)

    _input_df["SEX"] = _input_df["SEX"].replace({"male": 0, "female": 1})
    _input_df["MARRIAGE"] = _input_df["MARRIAGE"].replace({"single": 0, "married": 1, "other": 1})

    _input_df["BILL_AMT1_PER_LIMIT"] = _input_df["BILL_AMT1"] / _input_df["LIMIT_BAL"]
    _input_df["BILL_AMT2_PER_LIMIT"] = _input_df["BILL_AMT2"] / _input_df["LIMIT_BAL"]
    _input_df["BILL_AMT3_PER_LIMIT"] = _input_df["BILL_AMT3"] / _input_df["LIMIT_BAL"]
    _input_df["BILL_AMT4_PER_LIMIT"] = _input_df["BILL_AMT4"] / _input_df["LIMIT_BAL"]
    _input_df["BILL_AMT5_PER_LIMIT"] = _input_df["BILL_AMT5"] / _input_df["LIMIT_BAL"]
    _input_df["BILL_AMT6_PER_LIMIT"] = _input_df["BILL_AMT6"] / _input_df["LIMIT_BAL"]

    _input_df["PAY_PER_BILL_1"] = (_input_df["PAY_AMT1"] + 1) / (_input_df["BILL_AMT1"] + 1)
    _input_df["PAY_PER_BILL_2"] = (_input_df["PAY_AMT2"] + 1) / (_input_df["BILL_AMT2"] + 1)
    _input_df["PAY_PER_BILL_3"] = (_input_df["PAY_AMT3"] + 1) / (_input_df["BILL_AMT3"] + 1)
    _input_df["PAY_PER_BILL_4"] = (_input_df["PAY_AMT4"] + 1) / (_input_df["BILL_AMT4"] + 1)
    _input_df["PAY_PER_BILL_5"] = (_input_df["PAY_AMT5"] + 1) / (_input_df["BILL_AMT5"] + 1)
    _input_df["PAY_PER_BILL_6"] = (_input_df["PAY_AMT6"] + 1) / (_input_df["BILL_AMT6"] + 1)

    _input_df["TP_PER_TB"] = (_input_df["PAY_AMT1"] + _input_df["PAY_AMT2"] + _input_df["PAY_AMT3"] + _input_df["PAY_AMT4"] + _input_df["PAY_AMT5"] + _input_df["PAY_AMT6"] + 1) / (_input_df["BILL_AMT1"] + _input_df["BILL_AMT2"] + _input_df["BILL_AMT3"] + _input_df["BILL_AMT4"] + _input_df["BILL_AMT5"] + _input_df["BILL_AMT6"] + 1)

    _input_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    _input_df = _input_df.dropna()

    return _input_df

printh("h1", "Replace & Reformat Columns")

pd.set_option('future.no_silent_downcasting', True)
main_df = main_df.dropna()
main_df = add_features(main_df)

SELECT_COLUMNS = list(main_df.columns.values)
SELECT_FEATURES = [e for e in SELECT_COLUMNS if e != LABEL]
SELECT_COLUMNS = SELECT_FEATURES + [LABEL]
main_df = main_df[SELECT_COLUMNS].copy()
prep_columns = SELECT_COLUMNS
for prep_col in prep_columns:
    main_df[prep_col] = pd.to_numeric(main_df[prep_col], errors='raise')

printh("h2", "Main DF Info")
print(main_df.info())

printh("h2", "Described Columns")
print(main_df.describe())

In [ ]:
printh("h1", "Correlations")
show_corr(df=main_df)

In [ ]:
printh("h1", "Histograms")

printh("h2", "Skewness + Kurtosis")
show_skew_kurt(df=main_df)

# printh("h2", "Histograms")
# show_hists(df=main_df, features=SELECT_COLUMNS, n=5)

In [ ]:
printh("h1", "Scale Features & Label")

# cols = [LABEL]
cols_ex = ["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]
cols = [e for e in SELECT_FEATURES if e not in cols_ex]
print("Fix skew features %s" % (cols))
for col in cols:
    main_df[col] = np.cbrt(main_df[col])

cols = ["PAY_PER_BILL_1","PAY_PER_BILL_2","PAY_PER_BILL_3","PAY_PER_BILL_4","PAY_PER_BILL_5","PAY_PER_BILL_6"]
print("Fix skew features %s" % (cols))
for col in cols:
    main_df[col] = np.cbrt(main_df[col])

cols = [e for e in SELECT_FEATURES]
printh("h2", "Doing Scaling")
print("Scale features %s" % (cols))

main_df, SCALERS = scale(input_df=main_df, cols=cols, scaler_class=RobustScaler)
print(SCALERS)

# print("Update label values %s" % (LABEL))
# kbin = KBinsDiscretizer(n_bins=8, encode="ordinal", strategy="quantile")
# main_df[LABEL] = kbin.fit_transform(main_df[[LABEL]].copy())
# main_df[LABEL] = pd.to_numeric(main_df[LABEL], errors='coerce')

# printh("h2", "Label Vals (updated)")
# LABEL_VALS = sorted(main_df[LABEL].unique().tolist())
# print("main_df[\"%s\"].unique()" % (LABEL))
# print(LABEL_VALS)

main_df = main_df.dropna()

printh("h2", "Described Columns (updated)")
print(main_df.describe())

printh("h2", "Skewness + Kurtosis (updated)")
show_skew_kurt(df=main_df)

printh("h2", "Histograms (updated)")
show_hists(df=main_df, features=SELECT_COLUMNS, n=6)

printh("h2", "Correlations (updated)")
show_corr(df=main_df)

In [ ]:
printh("h1", "Split Data (train + test)")
# SELECT_FEATURES = []
# SELECT_COLUMNS = SELECT_FEATURES + [LABEL]
X = main_df[SELECT_FEATURES].to_numpy()
y = main_df[[LABEL]].to_numpy().reshape(-1)

# printh("h2", "Correlations (updated)")
# show_corr(df=main_df[SELECT_COLUMNS])

printh("h2", "All Data")
print(X.shape)
print(X[:1])
print(y.shape)
print(y[:10])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

printh("h2", "Train")
print(X_train.shape)
print(X_train[:1])
print(y_train.shape)
print(y_train[:10])

printh("h2", "Test")
print(X_test.shape)
print(X_test[:1])
print(y_test.shape)
print(y_test[:10])

In [ ]:
printh("h1", "Define Model")

model = Sequential([
    layers.Dense(100, activation='tanh', input_shape=(len(SELECT_FEATURES),)),
    layers.Dense(100, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0002),
    # loss=losses.MeanAbsoluteError(),
    loss=losses.BinaryCrossentropy(from_logits=False),
    metrics=['accuracy']
)

In [ ]:
printh("h1", "Start Model Training")

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=450,
    validation_split=0.1,
    verbose=2
)

In [ ]:
printh("h1", "Evaluate")

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy:.2f}")

loss, accuracy = model.evaluate(X_train, y_train, verbose=0)
print(f"Train Accuracy: {accuracy:.2f}")

In [ ]:
printh("h1", "Predict Samples")
prediction = model.predict(X_test[:10])
print(prediction)
if prediction.shape[1] > 1:
    # pred is 1-hot
    print(np.argmax(prediction, axis=1))
else:
    # pred is float 0~1
    print(np.round(prediction))

In [ ]:
printh("h1", "Create Confusion Mat")

# Make predictions
y_pred = model.predict(X_test)
if y_pred.shape[1] > 1:
    # pred is 1-hot
    y_pred = np.argmax(y_pred, axis=1)
else:
    # pred is float 0~1
    y_pred = np.round(y_pred)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')

# Plot the confusion matrix using Seaborn
conf_matrix = confusion_matrix(y_test, y_pred)
print(set(y_train.tolist()))
print(set(y_test.tolist()))
print(set(y_pred.flatten().tolist()))
cm_size = len(set(y_test.tolist()))
plt.figure(figsize=(cm_size, cm_size))
# sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=LABEL_VALS, yticklabels=LABEL_VALS)
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

printh("h1", "Describe Prediction Metrics")
y_pred = model.predict(X_test).flatten()
y_delta = abs(y_test - y_pred)
# print(y_delta)
y_delta_df = pd.DataFrame(data={
    "y_test": y_test,
    "y_pred": y_pred,
    "y_delta": y_delta,
})
desc = y_delta_df.describe()
desc["y_deltap"] = desc["y_delta"] / desc["y_test"]
print(desc)
print()

In [ ]:
printh("h1", "Training Info")

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy', color='blue')
plt.plot(history.history['val_accuracy'],
         label='Validation Accuracy', color='orange')
plt.title('Training and Validation Accuracy', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.title('Training and Validation Loss', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True)

plt.suptitle("Model Training Performance", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# printh("h1", "Feature Importances")

# def predict(X):
#     pred = model.predict(X)
#     if pred.shape[1] > 1:
#         # pred is 1-hot
#         pred = np.argmax(pred, axis=1)
#     else:
#         # pred is float 0~1
#         pred = np.round(pred)
#     return pred

# def accuracy(y_true, y_p):
#     return np.mean(y_true == y_p)

# def score(X, y):
#     y_pred = predict(X)
#     return accuracy(y, y_pred)

# printh("h2", "Get Feature Importances")
# base_score, score_decreases = get_score_importances(score, X, y)
# feature_importances = np.mean(score_decreases, axis=0)

# printh("h2", "Report")
# report = np.column_stack([SELECT_FEATURES, feature_importances])
# print(report.shape)
# print(report)
# report = np.array(sorted(report, key=lambda r: r[1], reverse=True))
# print(report.shape)
# print(report)

In [ ]:
# del(history)
# del(model)